In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_firm_structure_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "FirmStructureId string,"
        #     "SourceSystem string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "FirmStructureKey string,"
        #     "EmployeePartyKey string,"
        #     "EmployeeCategory string,"
        #     "EmployeeFirstName string,"
        #     "EmployeeLastName string,"
        #     "EmployeeFullName string,"
        #     "EmployeeEmail string,"
        #     "EmployeeUsername string,"
        #     "EmployeeUserTypeCode string,"
        #     "EmployeeUserType string,"
        #     "EmployeePositionCode string ,"
        #     "EmployeePosition string,"
        #     "DepartmentCode string,"
        #     "Department string,"
        #     "TeamCode string,"
        #     "Team string,"
        #     "SupervisorPartyKey string,"
        #     "SupervisorPartyName string,"
        #     "SupervisorUsername string,"
        #     "ManagerPartyKey string,"
        #     "ManagerPartyName string,"
        #     "ManagerUsername string,"
        #     "DirectorPartyKey string,"
        #     "DirectorPartyName string,"
        #     "DirectorUsername string,"
        #     "IsEmployeeActiveFlag boolean,"
        #     "CreatedDate date,"
        #     "LastUpdatedDate date,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.FirmStructure"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== employees ======================
        employees_extra_filters = None

        if employees_extra_filters is not None:
            employees_extra_filters.append(date_filter)
            employees_final_filters = " AND ".join(employees_extra_filters)
        else:
            employees_final_filters = date_filter

        employees_selected_fields = [
            "emp_code",
            "emp_first_name",
            "EMP_LAST_NAME",
            "emp_login",
            "EMP_USER_TYPE",
            "department",
            "DEP_CODE",
            "TEA_CODE",
            "TEA_DESCRIPTION",
            "supervisor_emp_code",
            "supervisor_login",
            "manager_emp_code",
            "manager_login",
            "EMP_ACTIVE",
            "EMP_CREATED",
            "updated",
            "businessdate",
        ]

        employees = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_employees",
            employees_final_filters,
            employees_selected_fields,
        )

        # ================== external_companies ======================
        external_companies_extra_filters = None

        if external_companies_extra_filters is not None:
            external_companies_extra_filters.append(date_filter)
            external_companies_final_filters = " AND ".join(
                external_companies_extra_filters
            )
        else:
            external_companies_final_filters = date_filter

        external_companies_selected_fields = [
            "EXTC_CODE",
            "EXTC_NAME",
            "EXTC_INTCODE",
            "EXTC_COMMENT",
            "BusinessDate",
        ]

        external_companies = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_external_companies",
            external_companies_final_filters,
            external_companies_selected_fields,
        )

        # ================== at_lawyers ======================
        lawyers_extra_filters = None

        if lawyers_extra_filters is not None:
            lawyers_extra_filters.append(date_filter)
            lawyers_final_filters = " AND ".join(lawyers_extra_filters)
        else:
            lawyers_final_filters = date_filter

        lawyers_selected_fields = [
            "LAWY_ID",
            "LAWY_FIRST_NAME",
            "LAWY_LAST_NAME",
            "LAWY_EMAIL",
            "BusinessDate",
        ]

        lawyers = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_lawyers",
            lawyers_final_filters,
            lawyers_selected_fields,
        )

        # ================ firm_structure_df_1 =================

        firm_structure_qca_primary_key_fields_1 = [
            employees.emp_code,
            lit("Employee"),
            lit("qca"),
        ]
        FirmStructureKey_primary_key_fields_1 = [employees.emp_code, lit("Employee")]

        sup = employees.select(
            col("emp_code").cast("string").alias("sup_code"),
            col("businessdate").cast("date").alias("sup_bd"),
            concat_ws(
                " ",
                coalesce(col("emp_first_name"), lit("")),
                coalesce(col("EMP_LAST_NAME"), lit("")),
            ).alias("sup_full_name"),
        ).alias("sup")

        mgr = employees.select(
            col("emp_code").cast("string").alias("mgr_code"),
            col("businessdate").cast("date").alias("mgr_bd"),
            concat_ws(
                " ",
                coalesce(col("emp_first_name"), lit("")),
                coalesce(col("EMP_LAST_NAME"), lit("")),
            ).alias("mgr_full_name"),
        ).alias("mgr")

        firm_structure_qca_1 = (
            (
                employees.withColumn(
                    "FirmStructureId",
                    sha2(
                        concat_ws(";", *firm_structure_qca_primary_key_fields_1),
                        256,
                    ),
                )
            )
            .withColumn(
                "FirmStructureKey",
                concat_ws(";", *FirmStructureKey_primary_key_fields_1),
            )
            .withColumn(
                "EmployeeFullName",
                concat_ws(
                    " ",
                    coalesce(employees["emp_first_name"], lit("")),
                    coalesce(employees["EMP_LAST_NAME"], lit("")),
                ),
            )
            .join(
                sup,
                (col("supervisor_emp_code").cast("string") == col("sup.sup_code"))
                & (col("businessdate").cast("date") == col("sup.sup_bd")),
                "left",
            )
            .join(
                mgr,
                (col("manager_emp_code").cast("string") == col("mgr.mgr_code"))
                & (col("businessdate").cast("date") == col("mgr.mgr_bd")),
                "left",
            )
        )

        firm_structure_qca_1 = firm_structure_qca_1.select(
            firm_structure_qca_1.FirmStructureId.cast("string").alias(
                "FirmStructureId"
            ),
            lit("qca").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_qca_1.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            employees.emp_code.cast("string").alias("EmployeePartyKey"),
            lit("Employee").alias("EmployeeCategory"),
            employees.emp_first_name.cast("string").alias("EmployeeFirstName"),
            employees.EMP_LAST_NAME.cast("string").alias("EmployeeLastName"),
            firm_structure_qca_1.EmployeeFullName.cast("string").alias(
                "EmployeeFullname"
            ),
            lit(None).alias("EmployeeEmail"),
            employees.emp_login.cast("string").alias("EmployeeUserName"),
            employees.EMP_USER_TYPE.cast("string").alias("EmployeeUserTypeCode"),
            employees.EMP_USER_TYPE.cast("string").alias("EmployeeUserType"),
            employees.department.cast("string").alias("EmployeePositionCode"),
            employees.department.cast("string").alias("EmployeePosition"),
            employees.DEP_CODE.cast("string").alias("DepartmentCode"),
            employees.department.cast("string").alias("Department"),
            employees.TEA_CODE.cast("string").alias("TeamCode"),
            employees.TEA_DESCRIPTION.cast("string").alias("Team"),
            employees.supervisor_emp_code.cast("string").alias("SupervisorPartyKey"),
            col("sup.sup_full_name").cast("string").alias("SupervisorPartyName"),
            employees.supervisor_login.cast("string").alias("SupervisorUserName"),
            employees.manager_emp_code.cast("string").alias("ManagerPartyKey"),
            col("mgr.mgr_full_name").cast("string").alias("ManagerPartyName"),
            employees.manager_login.cast("string").alias("ManagerUsername"),
            lit(None).cast("string").alias("DirectorUsername"),
            employees.EMP_ACTIVE.cast("boolean").alias("IsEmployeeActiveFlag"),
            employees.EMP_CREATED.cast("date").alias("CreatedDate"),
            employees.updated.cast("date").alias("LastUpdatedDate"),
            employees.businessdate.cast("date").alias("BusinessDate"),
        )

        # ================ firm_structure_df_2 =================

        firm_structure_qca_primary_key_fields_2 = [
            lawyers.LAWY_ID,
            lit("Lawyer"),
            lit("qca"),
        ]

        FirmStructureKey_primary_key_fields_2 = [lawyers.LAWY_ID, lit("Lawyer")]

        firm_structure_qca_2 = (
            lawyers.withColumn(
                "FirmStructureId",
                sha2(
                    concat_ws(";", *firm_structure_qca_primary_key_fields_2),
                    256,
                ),
            )
        ).withColumn(
            "FirmStructureKey", concat_ws(";", *FirmStructureKey_primary_key_fields_2)
        )

        firm_structure_qca_2 = firm_structure_qca_2.withColumn(
            "EmployeeFullName",
            concat_ws(
                " ",
                coalesce(lawyers["LAWY_FIRST_NAME"], lit("")),
                coalesce(lawyers["LAWY_LAST_NAME"], lit("")),
            ),
        )

        firm_structure_qca_2 = firm_structure_qca_2.select(
            firm_structure_qca_2.FirmStructureId.cast("string").alias(
                "FirmStructureId"
            ),
            lit("qca").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_qca_2.FirmStructureKey.alias("FirmStructureKey"),
            lawyers.LAWY_ID.alias("EmployeePartyKey"),
            lit("Lawyer").alias("EmployeeCategory"),
            lawyers.LAWY_FIRST_NAME.alias("EmployeeFirstName"),
            lawyers.LAWY_LAST_NAME.alias("EmployeeLastName"),
            firm_structure_qca_2.EmployeeFullName.alias("EmployeeFullname"),
            lawyers.LAWY_EMAIL.alias("EmployeeEmail"),
            lit(None).alias("EmployeeUserName"),
            lit(None).alias("EmployeeUserTypeCode"),
            lit(None).alias("EmployeeUserType"),
            lit(None).alias("EmployeePositionCode"),
            lit(None).alias("EmployeePosition"),
            lit(None).alias("DepartmentCode"),
            lit(None).alias("Department"),
            lit(None).alias("TeamCode"),
            lit(None).alias("Team"),
            lit(None).alias("SupervisorPartyKey"),
            lit(None).alias("SupervisorPartyName"),
            lit(None).alias("SupervisorUserName"),
            lit(None).alias("ManagerPartyKey"),
            lit(None).alias("ManagerPartyName"),
            lit(None).alias("ManagerUsername"),
            lit(None).alias("DirectorUsername"),
            lit(None).alias("IsEmployeeActiveFlag"),
            lit(None).alias("CreatedDate"),
            lit(None).alias("LastUpdatedDate"),
            lawyers.BusinessDate.alias("BusinessDate"),
        )

        # ================ firm_structure_df_3 =================

        firm_structure_qca_primary_key_fields_3 = [
            external_companies.EXTC_CODE,
            lit("External Company"),
            lit("qca"),
        ]

        FirmStructureKey_primary_key_fields_3 = [
            external_companies.EXTC_CODE,
            lit("External Company"),
        ]

        firm_structure_qca_3 = (
            external_companies.withColumn(
                "FirmStructureId",
                sha2(
                    concat_ws(";", *firm_structure_qca_primary_key_fields_3),
                    256,
                ),
            )
        ).withColumn(
            "FirmStructureKey",
            concat_ws(";", *FirmStructureKey_primary_key_fields_3),
        )

        firm_structure_qca_3 = firm_structure_qca_3.select(
            firm_structure_qca_3.FirmStructureId.cast("string").alias(
                "FirmStructureId"
            ),
            lit("qca").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_qca_3.FirmStructureKey.alias("FirmStructureKey"),
            external_companies.EXTC_CODE.alias("EmployeePartyKey"),
            lit("External Company").alias("EmployeeCategory"),
            lit(None).alias("EmployeeFirstName"),
            lit(None).alias("EmployeeLastName"),
            external_companies.EXTC_NAME.alias("EmployeeFullname"),
            lit(None).alias("EmployeeEmail"),
            lit(None).alias("EmployeeUserName"),
            external_companies.EXTC_INTCODE.cast("string").alias("EmployeeUserTypeCode"),
            external_companies.EXTC_COMMENT.cast("string").alias("EmployeeUserType"),
            lit(None).alias("EmployeePositionCode"),
            lit(None).alias("EmployeePosition"),
            lit(None).alias("DepartmentCode"),
            lit(None).alias("Department"),
            lit(None).alias("TeamCode"),
            lit(None).alias("Team"),
            lit(None).alias("SupervisorPartyKey"),
            lit(None).alias("SupervisorPartyName"),
            lit(None).alias("SupervisorUserName"),
            lit(None).alias("ManagerPartyKey"),
            lit(None).alias("ManagerPartyName"),
            lit(None).alias("ManagerUsername"),
            lit(None).alias("DirectorUsername"),
            lit(None).alias("IsEmployeeActiveFlag"),
            lit(None).alias("CreatedDate"),
            lit(None).alias("LastUpdatedDate"),
            external_companies.BusinessDate.alias("BusinessDate"),
        )

        # ================== Append ==================
        firm_structure_qca_4 = firm_structure_qca_2.union(firm_structure_qca_1)

        Result_df = firm_structure_qca_4.union(firm_structure_qca_3)

        Result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

        logging.info("We did it!")


In [0]:
load_curated_firm_structure_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)